In [1]:
from google.colab import drive # for CUDA
drive.mount('/content/drive', force_remount=True)
import pandas as pd
import shutil
import os
from pathlib import Path
import zipfile

Mounted at /content/drive


In [2]:
################################ L0CAL DATASET #################################
# speeds up image fetching, instead of fetching from Drive each new session
#A7

# copying existing zip Dataset file to speed up image fetching process
print(f"Copying zip folder from drive to local colab storage")
!cp "/content/drive/My Drive/Colab Notebooks/FinalProject/Dataset.zip" /content/Dataset.zip

print(f"Copied. Now unzipping the folder")
!unzip -q /content/Dataset.zip -d /content/Dataset #q = Quiet so it doesnt print out each image name
print(f"Unzipped.")

# checking amounts of images loaded in
# find -type f looks for every single file
# wc - l = Word Count - Lines , counts the lines and prints as a nr instead of individual img names
print(f"Amount of train images in local storage: (should be 140002)")
!find /content/Dataset/Train -type f | wc -l

print(f"Amount of val images in local storage: (should be 39428)")
!find /content/Dataset/Validation -type f | wc -l

print(f"Amount of test images in local storage: (should be 10905)")
!find /content/Dataset/Test -type f | wc -l

# deleting zip file from local storage while keeping unzipped Dataset
# to take back 1.68 GB of disk space
!rm /content/Dataset.zip

Copying zip folder from drive to local colab storage
Copied. Now unzipping the folder
Unzipped.
Amount of train images in local storage: (should be 140002)
140002
Amount of val images in local storage: (should be 39428)
39428
Amount of test images in local storage: (should be 10905)
10905


In [3]:
# A7
preds_checkpoints = "/content/drive/My Drive/Colab Notebooks/FinalProject/Checkpoints/"

csv_files = {
    "ANY_MISCLASSIFIED": f"{preds_checkpoints}any_misclassifications.csv",
    "ALL_MISCLASSIFIED": f"{preds_checkpoints}all_misclassified.csv",
    "OWN_MISCLASSIFIED": f"{preds_checkpoints}own_misclassified.csv",
    "PT_MISCLASSIFIED": f"{preds_checkpoints}pt_misclassified.csv",
}

for folder_name, csv_path in csv_files.items():
    df = pd.read_csv(csv_path)
    output_folder = Path(f"/content/Findings/{folder_name}")
    output_folder.mkdir(parents=True, exist_ok=True)
    for path in df["path"]:
        shutil.copy(path, output_folder) # copies only the specific images from "path" in each csv
    print(f"Done: {folder_name}")

Done: ANY_MISCLASSIFIED
Done: ALL_MISCLASSIFIED
Done: OWN_MISCLASSIFIED
Done: PT_MISCLASSIFIED


In [4]:

# zipping each folder in local memory for speed
for folder_name in csv_files.keys(): #goes through each file in csv_files above
    folder_path = Path(f"/content/Findings/{folder_name}") #path to local data
    zip_path = f"/content/{folder_name}.zip" #path where zips will save
    with zipfile.ZipFile(zip_path, 'w') as zipf:
        for file in folder_path.iterdir():
            zipf.write(file, file.name) #writes each image into the zip using the filename
    print(f"Zipped: {folder_name}") #for confirmation

# copying the zipped folders to Drive
output_dir = "/content/drive/My Drive/Colab Notebooks/FinalProject/Findings/"
os.makedirs(output_dir, exist_ok=True) #doesnt crash if the directory already exists if code is rerun

for folder_name in csv_files.keys():
    zip_path = f"/content/{folder_name}.zip"
    shutil.copy(zip_path, output_dir) # copying the zip to the path folder on Drive
    print(f"Copied to Drive: {folder_name}") #again, for confirmation


Zipped: ANY_MISCLASSIFIED
Zipped: ALL_MISCLASSIFIED
Zipped: OWN_MISCLASSIFIED
Zipped: PT_MISCLASSIFIED
Copied to Drive: ANY_MISCLASSIFIED
Copied to Drive: ALL_MISCLASSIFIED
Copied to Drive: OWN_MISCLASSIFIED
Copied to Drive: PT_MISCLASSIFIED


In [5]:
for folder_name, csv_path in csv_files.items():
    df = pd.read_csv(csv_path)
    print(f"{folder_name}: {len(df)} rows")

ANY_MISCLASSIFIED: 2244 rows
ALL_MISCLASSIFIED: 138 rows
OWN_MISCLASSIFIED: 641 rows
PT_MISCLASSIFIED: 249 rows
